# 10 — Deconstruction report (paper §6-§7)

**Read-only report** over the canonical artefacts (2026-06-12 VM run).
Narrative: the naive quantile-LP *appears* to show downside amplification (the bait, §6) — the diagnostics show it is a genuine but **symmetric** volatility effect, generic to crypto stress, misread as downside-specificity (§7).

In [ ]:
# Setup — read-only report over canonical artefacts (data/econ, 2026-06-12)
import sys
from pathlib import Path
ROOT = Path.cwd().parent if (Path.cwd() / "..").resolve().name else Path.cwd()
ROOT = Path.cwd().parent  # notebooks/ -> repo root
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts" / "paper"))
sys.path.insert(0, str(ROOT / "scripts"))

import pandas as pd
pd.set_option("display.width", 140)
import style, make_figures as MF
style.apply()
from config import ECON_DIR
print("canonical dir:", ECON_DIR)

## 1. The bait — naive QLP IRF (Fig 3)
β(0.01, h=0) = −0.032 → −0.241 at h=12 (≈19× the median at impact); the median barely moves. Band: block-bootstrap on the exact main-table spec.

In [ ]:
qlp = pd.read_csv(ECON_DIR / "quantile_lp_results.csv")
qlp[qlp.h.isin([0,1,3,6,12,24])].pivot_table(index="tau", columns="h", values="beta_shock").round(3)

In [ ]:
MF.fig3_qlp_irf()

## 2. The pillar — symmetric sign-flip placebo (Fig 4)
A **zero-skew** world (Rademacher sign-flip on real magnitudes; the empirical volatility path is preserved exactly, no vol model) **reproduces** the left-right gap at every horizon: h=0 inside the band (no genuine impact gap), h=12 dead-centre. The vol-model-scaled variant (rolling + GARCH, genuinely distinct after the σ-cancellation fix) agrees 14/14 cells.

In [ ]:
pl = pd.read_csv(ECON_DIR / "placebo_symmetric.csv")
pl[pl.tau.isin([0.01, 0.99])].round(4)

In [ ]:
MF.fig4_placebo_gap_distribution()

## 3. The dual pure-null (Fig 9)
Both principled nulls give **small** artifact shares (circular-shift 0.18, innovation-shuffle 0.08 at h=12), **no signed bias**, and the real β outside the null band at every horizon (permutation p ≤ .038). The historical “72%” is reproducible under neither — the long-horizon β is genuine; the *interpretation* was the artifact.

In [ ]:
circ = pd.read_csv(ECON_DIR / "pure_null_circular_shift_by_horizon.csv")
innov = pd.read_csv(ECON_DIR / "pure_null_innov_shuffle_by_horizon.csv")
pd.concat([circ.assign(null="circular_shift"), innov.assign(null="innov_shuffle")])[["null","h","true_beta","artifact_beta_mean","ratio","artifact_beta_signed_mean","perm_pval_abs"]].round(4)

In [ ]:
MF.fig9_pure_null_dual()

## 4. Overlap re-creates the fake gap (Fig 5, right panel)
The clean per-period exceedance asymmetry Δ is null at every h; the **cumulative (overlapping)** variant manufactures a significant Δ at h=12 (p=0.023) — the artifact made visible in one picture.

In [ ]:
MF.fig5_exceedance_symmetry()

## 5. Not ETH-specific — the BTC-outcome placebo (Fig 8)
The full QLP re-run with **BTC as outcome** (mirror controls) reproduces the signature at 0.56–0.75× the ETH magnitude with the same deepening: the pattern is **generic crypto stress**, not a DeFi-ETH channel. (Kernel sanity: re-fitted ETH cells match the canonical table at ~1e-17.)

In [ ]:
prof = pd.read_csv(ECON_DIR / "btc_vs_eth_profile.csv")
prof = prof[prof.h.astype(str) != "h"].astype(float)
prof[prof.h.isin([0,6,12,24])].round(3)

In [ ]:
MF.fig8_btc_vs_eth()